#### Name generation based on Transformer

mask creation test

    import torch

    scores = torch.randn(4, 4)
    print("score: ", scores)

    tril = torch.tril(torch.ones(4, 4), diagonal=0)
    print("tril: ", tril)

    scores = scores.masked_fill(tril == 0, float("-inf"))
    print("scores: ", scores)

#### Transformer Generator

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from xd_masked_selfattention import XD_TransformerDecoderBlock

class XD_TransformerGenerator(nn.Module):
  # voca_size : 임베딩할 고유 인덱스의 개수 (예: 단언 사전 크기)
  # string 이라면, string 을 구성하는 character 사전 크기
  def __init__(self, voca_size, embed_dim, num_heads, num_layers, block_size):
    super().__init__()

    # attention layer가 몇 개의 글자까지 볼 수 있는지를 나타냄.
    self.block_size = block_size

    # Embedding : 정수 인덱스를 연속적인 임베딩벡터로 변환
    # num_embeddings : 임베딩할 고유 인덱스의 개수 (예: 단언 사전 크기)
    # embed_dim : 각 인덱스를 나타낼 임베딩 벡터 차원
    self.char_embedding = nn.Embedding(voca_size, embed_dim)

    # Positional Encoding (character의 위치 vector)
    self.pos_embedding = nn.Embedding(block_size, embed_dim)

    # Transformer decoder blocks
    self.transformer_blocks = nn.Sequential(*[XD_TransformerDecoderBlock(embed_dim, num_heads) for _ in range(num_layers)])

    # last layer normalization
    self.last_lnorm = nn.LayerNorm(embed_dim)

    # 최종 출력 선형 변환
    self.last_fc = nn.Linear(embed_dim, voca_size)


  def forward(self, x):
    # name's char embedding vector (with char unit)
    # char_embeddings = [batch_size, char_length, embed_dim] = [batch_size, seq_length, embed_dim]
    char_embeddings = self.char_embedding(x)  

    # name's character position
    # positions = [1, char_length] = [1, seq_length]
    char_positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)

    # position embedding vector = [1, char_length, embed_dim] = [1, seq_legnth, embed_dim]
    pos_embeddings = self.pos_embedding(char_positions)

    # 임베딩 벡터와 위치 벡터를 더함
    x = char_embeddings + pos_embeddings

    # Transformer Decoder Blocks
    x = self.transformer_blocks(x)

    # last layer normalization
    x = self.last_lnorm(x)

    # last linear transformation
    logits = self.last_fc(x)

    return logits
  

  """
    transformer decoder logits 결과를 이용하여 string 생성
  """
  def generate(self, indices, max_length=100):
    with torch.no_grad():
      for _ in range(max_length):
        # block size 만큼 indices 를 얻어옴
        idx_block = indices[:, -self.block_size:]
        # forward pass : logits = [batch_size, block_size, voca_size]
        logits = self(idx_block)
        # infered last character 만 필요함
        logits = logits[:, -1, :]

        # logits -> probability 
        probs = F.softmax(logits, dim=-1)
        # sampling based on probability
        idx_next = torch.multinomial(probs, num_samples=1)
        # update indices
        indices = torch.cat([indices, idx_next], dim=-1)

    return indices




In [2]:
torch.manual_seed(1337)

# embedding vector length 
embed_dim = 32
head_count = 4
# attention dim : embed_dim // 4 = 8

layer_count = 4
# 몇 개의 글자까지 관계를 볼 것인지를 나타냄.
block_size = 16

#### Read a bible text 

In [3]:
with open("bible.txt", encoding="utf-8") as f:
  text = f.read()

print(text[:100])

KJV
King James Bible: Pure Cambridge Edition - Text courtesy of www.BibleProtector.com
Genesis 1:1	I


#### Character Bag (Voca)

In [4]:
# all characters in bible file
chars = sorted(list(set(text)))
char_voca_size = len(chars)
print(''.join(chars))
print(f"charecter voca size: {char_voca_size}")

	
 !(),-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWYZ[]abcdefghijklmnopqrstuvwxyz—’
charecter voca size: 77


#### Encode, Decode function

In [5]:
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[ch] for ch in s]
decode = lambda e: ''.join([itos[i] for i in e])

print(f"{encode("nocope deeplearning")}")
print(f"{decode(encode("nocope deeplearning"))}")

[62, 63, 51, 63, 64, 53, 2, 52, 53, 53, 64, 60, 53, 49, 66, 62, 57, 62, 55]
nocope deeplearning


In [6]:
encoded_text = encode(text)
data = torch.tensor(encoded_text, dtype=torch.long)

print(f"text[0:100]: {text[0:100]}\n")
print(f"data[0:100]: {data[0:100]}")


text[0:100]: KJV
King James Bible: Pure Cambridge Edition - Text courtesy of www.BibleProtector.com
Genesis 1:1	I

data[0:100]: tensor([32, 31, 43,  1, 32, 57, 62, 55,  2, 31, 49, 61, 53, 67,  2, 23, 57, 50,
        60, 53, 19,  2, 37, 69, 66, 53,  2, 24, 49, 61, 50, 66, 57, 52, 55, 53,
         2, 26, 52, 57, 68, 57, 63, 62,  2,  7,  2, 41, 53, 72, 68,  2, 51, 63,
        69, 66, 68, 53, 67, 73,  2, 63, 54,  2, 71, 71, 71,  8, 23, 57, 50, 60,
        53, 37, 66, 63, 68, 53, 51, 68, 63, 66,  8, 51, 63, 61,  1, 28, 53, 62,
        53, 67, 57, 67,  2, 10, 19, 10,  0, 30])


#### Get a random batch of the data

In [7]:
batch_size = 4

# block_size : 몇 개의 글자까지 관계를 볼 것인지를 나타냄.
def get_batch(data, batch_size, block_size):
  # random choice a block_size indices
  indices = torch.randint(len(data)-block_size, (batch_size,))
  # get a batch data
  batch_input = torch.stack([data[i:i+block_size] for i in indices])
  batch_output = torch.stack([data[i+1:i+block_size+1] for i in indices])
  return batch_input, batch_output

batch_input, batch_output = get_batch(data, batch_size, block_size)

print(f"batch_input: {batch_input}")
print(f"batch_output: {batch_output}")

batch_input: tensor([[60, 52,  8,  1, 37, 67, 49, 60, 61,  2, 13, 13, 19, 11,  0, 47],
        [49, 67, 68,  2, 68, 49, 59, 53, 62,  2, 61, 73,  2, 56, 69, 67],
        [ 2, 57, 54,  2, 56, 53,  2, 56, 49, 70, 53,  2, 50, 53, 68, 66],
        [ 2, 56, 57, 61,  2, 40, 53, 55, 69, 50,  8,  1, 10,  2, 24, 56]])
batch_output: tensor([[52,  8,  1, 37, 67, 49, 60, 61,  2, 13, 13, 19, 11,  0, 47, 29],
        [67, 68,  2, 68, 49, 59, 53, 62,  2, 61, 73,  2, 56, 69, 67, 50],
        [57, 54,  2, 56, 53,  2, 56, 49, 70, 53,  2, 50, 53, 68, 66, 63],
        [56, 57, 61,  2, 40, 53, 55, 69, 50,  8,  1, 10,  2, 24, 56, 66]])


#### Training steps

Torch Device

In [8]:
print(torch.__version__)

if torch.backends.mps.is_available():
  my_device = torch.device('mps')
elif torch.cuda.is_available():
  my_device = torch.device('cuda')
else:
  my_device = torch.device('cpu')

print(my_device)

2.7.0+cu126
cuda


In [9]:
import torch.optim as optim

# training parameters
learning_rate = 1e-3
epochs = 1000000

# model creation
model = XD_TransformerGenerator(voca_size=char_voca_size, embed_dim=embed_dim, num_heads=head_count, 
                              num_layers=layer_count, block_size=block_size)
model.to(my_device)
# model.train()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)

batch_size = 4
total_loss = 0

# Training
for epoch in range(epochs):
  # get a random batch data
  batch_input, batch_output = get_batch(data, batch_size=batch_size, block_size=block_size)
  # to device tensor
  batch_input = batch_input.to(my_device)
  batch_output = batch_output.to(my_device)

  # zero the gradients
  optimizer.zero_grad()

  # inference : forward pass
  logits = model(batch_input)

  # view(-1, logits.size(-1)) : 2D (batch_size * block_size, voca_size)
  # voca_size 는 각각 character probability 를 나타냄.
  logits = logits.view(-1, logits.size(-1))
  # view(-1) : 1D (batch_size * block_size,)
  batch_output = batch_output.view(-1)


  # compute loss
  loss = F.cross_entropy(logits, batch_output)
  total_loss += loss.item()

  # backward pass : compute gradient of the loss with respect to model parameters
  loss.backward()
  # step function : update the weights
  optimizer.step()

  # 1000 epoch 마다 생성된 문장 출력
  if epoch % 1000 == 0:
    print(f"epoch {epoch} : loss {loss.item()}")

    # 첫 글자 생성
    first_index = torch.zeros((1, 1), dtype=torch.long).to(my_device)
    # 최대 500 글자 생성
    generated_text = model.generate(indices=first_index, max_length=100)
    # 생성된 문장 디코딩
    decoded_text = decode(generated_text[0].tolist())
    # 생성된 문장 출력
    print(f"generated text: {decoded_text}")


print(f"total loss: {total_loss / epochs}")


epoch 0 : loss nan


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
